# QSVM — Experiment 9 (learning curves), redone with leakage-free sampling

**Question.** Does the advantage of non-linear kernels (RBF, polynomial, and the quantum feature maps) over a linear kernel
grow as the training set shrinks? (Dr. Broderick's hypothesis, and Dr. Rayan's point 9.) Everything here uses **all five models** (ZZFeatureMap, ZFeatureMap, linear, RBF, polynomial), with the representation
chosen in the inner loop from the same catalogue as the nested notebook (Run B).
The 45-value table is **not** used: it cannot be run on the quantum kernels (45 qubits).

## What is kept identical to the final nested experiment (`QSVM_Steve_Notes_nested.ipynb`)
Outer loop: 5 folds x 20 repeats, **group-aware**, same seeds -> the same 100 outer test folds (16 peptides each).
Inner loop: **4** group-aware stratified folds, same `random_state`. Same hyperparameter grids, same selection metric
(balanced accuracy), a new `MinMaxScaler` fitted on the training part only, same exact statevector kernel, same intervals
(Nadeau-Bengio and group bootstrap). At the largest size (the whole outer-training fold) this notebook reproduces the nested
notebook's predictions **exactly** in all 100 folds, for all five models (checked, see the last markdown cell).

## What changed, and why it does not cost statistical validity
| | old Experiment 9 | this notebook |
|---|---|---|
| how the training set is shrunk | whole sequence groups were drawn until the count reached `size` | **individual peptides**, class-stratified, drawn **only from the outer-training fold** |
| actual training size | overshoots (e.g. 16.9 on average for "16") | **exact** (16, 24, ..., then the whole fold, about 64) |
| subsets across sizes | independent draws per size | **nested prefixes** of one random order per outer fold (16 < 24 < ... < full) |
| inner CV at small n | at n = 16, 36 of 100 outer folds had an inner fold with a single class | 1 of 100 at n = 16; unusable inner folds are skipped, never the outer fold |
| dropped outer folds | folds could be silently skipped | **none**: all 100 folds at every size; a flagged fallback exists but was never triggered |
| outer repeats | 10 | 20 |

* **Group-awareness is preserved.** Group leakage is only a train/test problem. The training subset is a subset of an outer-training
  fold that is already group-disjoint from the outer test fold, so no group can reach both sides (asserted inside the loop).
* **The outer test fold never changes with the training size**, so every size, model and representation is evaluated on identical
  test peptides (paired).
* **The inner loop is the same at every size.** Only the amount of data changes; the procedure being assessed does not.
* **Why not 5 outer x 3 inner?** The inner K is part of the procedure under assessment. Changing it for the curve would make the last
  point differ from the main experiment. It also does not fix the real problem: with the new sampling K = 4 is already almost never degenerate.
  If you still want K = 3, set `N_SPLITS_INNER = 3` here **and** in the nested notebook, never in only one of them.

In [1]:
import os, json, hashlib
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
import itertools, time, warnings
import numpy as np, pandas as pd
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVC
from sklearn.metrics import (balanced_accuracy_score, accuracy_score,
                             matthews_corrcoef, confusion_matrix)
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel, polynomial_kernel

RANDOM_SEED = 42
N_SPLITS_OUTER, N_REPEATS_OUTER, N_SPLITS_INNER = 5, 20, 4   # same as the main nested experiment
ENTANGLEMENT = 'linear'
PARAM_GRID_QUANTUM = {'reps': [1, 2], 'bandwidth': [0.05, 0.1, 0.2, 0.5], 'C': [0.1, 1, 10]}
CLASSICAL_GRIDS = {
    'linear': {'C': [0.1, 1, 10]},
    'rbf':    {'C': [0.1, 1, 10], 'gamma': ['scale', 0.01, 0.1, 1]},
    'poly':   {'C': [0.1, 1, 10], 'degree': [2, 3], 'gamma': ['scale', 0.1, 1], 'coef0': [0.0, 1.0]},
}
# used ONLY if no inner fold is usable (never silently drops an outer fold)
FALLBACK_CLASSICAL = {'linear': {'C': 1}, 'rbf': {'C': 1, 'gamma': 'scale'},
                      'poly': {'C': 1, 'degree': 3, 'gamma': 'scale', 'coef0': 1.0}}
FALLBACK_QUANTUM = {'reps': 1, 'bandwidth': 0.1, 'C': 1}
TRAIN_SIZES = [16, 24, 32, 40, 48, 56, 'full']   # 'full' = the whole outer-training fold (~64)

OUTDIR = 'results_exp9_final'
os.makedirs(OUTDIR, exist_ok=True)
RUN_MODE_B = True   # representation selected in the inner loop (heavier; see section 8)
print(f'outer {N_SPLITS_OUTER}x{N_REPEATS_OUTER} = {N_SPLITS_OUTER*N_REPEATS_OUTER} folds, inner {N_SPLITS_INNER}, sizes {TRAIN_SIZES}')

outer 5x20 = 100 folds, inner 4, sizes [16, 24, 32, 40, 48, 56, 'full']


In [2]:
import os, json, hashlib
import itertools, time, warnings
import numpy as np, pandas as pd
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVC
from sklearn.metrics import (balanced_accuracy_score, accuracy_score,
                             matthews_corrcoef, confusion_matrix)
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel, polynomial_kernel

RANDOM_SEED = 42
N_SPLITS_OUTER, N_REPEATS_OUTER, N_SPLITS_INNER = 5, 20, 4   # same as the main nested experiment
ENTANGLEMENT = 'linear'
PARAM_GRID_QUANTUM = {'reps': [1, 2], 'bandwidth': [0.05, 0.1, 0.2, 0.5], 'C': [0.1, 1, 10]}
CLASSICAL_GRIDS = {
    'linear': {'C': [0.1, 1, 10]},
    'rbf':    {'C': [0.1, 1, 10], 'gamma': ['scale', 0.01, 0.1, 1]},
    'poly':   {'C': [0.1, 1, 10], 'degree': [2, 3], 'gamma': ['scale', 0.1, 1], 'coef0': [0.0, 1.0]},
}
# used ONLY if no inner fold is usable (never silently drops an outer fold)
FALLBACK_CLASSICAL = {'linear': {'C': 1}, 'rbf': {'C': 1, 'gamma': 'scale'},
                      'poly': {'C': 1, 'degree': 3, 'gamma': 'scale', 'coef0': 1.0}}
FALLBACK_QUANTUM = {'reps': 1, 'bandwidth': 0.1, 'C': 1}
TRAIN_SIZES = [16, 24, 32, 40, 48, 56, 'full']   # 'full' = the whole outer-training fold (~64)

OUTDIR = 'results_exp9_final'
os.makedirs(OUTDIR, exist_ok=True)
RUN_MODE_B = True   # representation selected in the inner loop (heavier; see section 8)
print(f'outer {N_SPLITS_OUTER}x{N_REPEATS_OUTER} = {N_SPLITS_OUTER*N_REPEATS_OUTER} folds, inner {N_SPLITS_INNER}, sizes {TRAIN_SIZES}')

outer 5x20 = 100 folds, inner 4, sizes [16, 24, 32, 40, 48, 56, 'full']


## 2. Labels, sequences and representations

In [3]:
REPRESENTATIONS = {   # the same catalogue as the nested notebook; full45 is deliberately absent
    'mean_z':               'zscale_dataset.csv',
    'z1_positional':        'zscaled_z1_only.csv',
    'pc1_positional_87':    'PC1_data_frame_87.csv',
    'hopp_woods':           'hopp_woods.csv',
    'kyte_doolittle':       'kyte_doolittle.csv',
    'fauchere_pliska':      'fauchere_pliska.csv',
    'euclid_centroid':      'euclid_centroid.csv',
    'mahalanobis_centroid': 'mahalanobis_centroid.csv',
}
LABEL_LIKE_NAMES = {'y', 'label', 'class', 'target', 'binding_energy', 'energy'}


def load_representation(csv_path, sequences, y):
    data = pd.read_csv(csv_path)
    if 'aa1' in data.columns:                      # per-amino-acid scale table -> one feature per position
        value_cols = [c for c in data.columns if c not in ('aa1', 'aa3')]
        if len(value_cols) != 1:
            raise ValueError(f'{csv_path}: expected exactly one value column, got {value_cols}')
        lookup = dict(zip(data['aa1'], data[value_cols[0]]))
        missing = set(''.join(sequences)) - set(lookup)
        if missing:
            raise ValueError(f'{csv_path}: no scale value for residues {missing}')
        return np.array([[lookup[r] for r in seq] for seq in sequences], dtype=float)
    data = data.drop(columns=[c for c in data.columns if not pd.api.types.is_numeric_dtype(data[c])])
    drop = [c for c in data.columns
            if c.strip().lower() in LABEL_LIKE_NAMES
            or (set(np.unique(data[c].values)) <= {0, 1} and np.array_equal(data[c].values.astype(int), y))]
    return data.drop(columns=drop).values.astype(float)


_lab = pd.read_excel('Docking_high_low_energy_labels.xlsx', sheet_name='Results')
_lab = _lab.dropna(subset=['Sequence Epitope)']).reset_index(drop=True)
sequences = _lab['Sequence Epitope)'].astype(str).str.strip().tolist()
y = (_lab['Otsu Class theshold -77.8'] == 'Strong').astype(int).values

representations = {}
for name, path in REPRESENTATIONS.items():
    if not os.path.exists(path):
        print(f'   (skipping {name}: {path} not found -- it will simply not be a candidate in section 10)')
        continue
    X = load_representation(path, sequences, y)
    assert X.shape[0] == len(y) and np.isfinite(X).all(), name
    representations[name] = X
assert 'mean_z' in representations, 'zscale_dataset.csv is required'
print(f'{len(y)} peptides, class balance {np.bincount(y)}')
print({k: v.shape[1] for k, v in representations.items()}, '(features = qubits)')

# consistency check: mean_z must equal the mean over the 9 positions of the 45-value table
if os.path.exists('zscaled_5_45.csv'):
    f45 = pd.read_csv('zscaled_5_45.csv', index_col=0).values
    z_chk = np.stack([f45[:, [p * 5 + k for p in range(9)]].mean(axis=1) for k in range(5)], axis=1)
    diff = np.abs(z_chk - representations['mean_z']).max()
    print(f'max |zscale_dataset.csv - mean of zscaled_5_45.csv| = {diff:.2e}', '' if diff < 1e-6 else '  <-- WARNING: the two do not agree')

80 peptides, class balance [42 38]
{'mean_z': 5, 'z1_positional': 9, 'pc1_positional_87': 9, 'hopp_woods': 9, 'kyte_doolittle': 9, 'fauchere_pliska': 9, 'euclid_centroid': 9, 'mahalanobis_centroid': 9} (features = qubits)


/home/luispabloelchido/miniconda3/envs/QML/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


## 3. Sequence-redundancy groups (identical to the nested notebook)

In [4]:
K_SHARED, IDENTITY_THRESH = 5, 7 / 9

def similarity_edges(seqs, k_shared=K_SHARED, identity_thresh=IDENTITY_THRESH):
    n, L = len(seqs), len(seqs[0])
    kmers = [{s[i:i + k_shared] for i in range(L - k_shared + 1)} for s in seqs]
    return [(i, j) for i in range(n) for j in range(i + 1, n)
            if (kmers[i] & kmers[j]) or sum(a == b for a, b in zip(seqs[i], seqs[j])) / L >= identity_thresh]

edges = similarity_edges(sequences)
n_cc, groups = connected_components(coo_matrix((np.ones(len(edges)), ([e[0] for e in edges], [e[1] for e in edges])),
                                               shape=(len(y), len(y))), directed=False)
print(f'{len(edges)} similarity edges -> {n_cc} groups (the report states 56 edges and 43 groups)')

56 similarity edges -> 43 groups (the report states 56 edges and 43 groups)


## 4. Exact fidelity kernel and helpers (copied unchanged from the nested notebook)

In [5]:
# ---------------------------------------------------------------- UNCHANGED: analytic statevectors
def _fwht_inplace(a):
    N, dim = a.shape
    h = 1
    while h < dim:
        v = a.reshape(N, -1, 2, h)
        x, yy = v[:, :, 0, :], v[:, :, 1, :]
        t = x - yy
        x += yy
        yy[...] = t
        h *= 2
    return a

def statevectors(X, kind, reps, entanglement=ENTANGLEMENT):
    X = np.asarray(X, dtype=float)
    N, n = X.shape
    dim = 2 ** n
    if kind not in ('z', 'zz'):
        raise ValueError(kind)
    if kind == 'zz' and entanglement != 'linear':
        raise ValueError("only 'linear' entanglement is implemented")
    single = np.empty((n, N, 2), dtype=complex)
    single[:, :, 0] = 1.0
    single[:, :, 1] = np.exp(1j * 2.0 * X.T)
    if kind == 'zz':
        pair = np.ones((n - 1, N, 2, 2), dtype=complex)
        for i in range(n - 1):
            e = np.exp(1j * 2.0 * (X[:, i] - np.pi) * (X[:, i + 1] - np.pi))
            pair[i, :, 0, 1] = e
            pair[i, :, 1, 0] = e
    acc = single[n - 1]
    for i in range(n - 2, -1, -1):
        acc = acc.reshape(N, -1, 2)
        new = acc[:, :, :, None] * single[i][:, None, None, :]
        if kind == 'zz':
            new = new * pair[i][:, None, :, :]
        acc = new.reshape(N, -1)
    diag = acc
    state = diag * (dim ** -0.5)
    for _ in range(reps - 1):
        state = _fwht_inplace(state) * (dim ** -0.5)
        state *= diag
    return state

def kernel_train_test(X_tr, X_te, kind, reps, entanglement=ENTANGLEMENT):
    sv = statevectors(np.vstack([X_tr, X_te]), kind, reps, entanglement)
    m = len(X_tr)
    return np.abs(sv[:m].conj() @ sv[:m].T) ** 2, np.abs(sv[m:].conj() @ sv[:m].T) ** 2


# ---------------------------------------------------------------- UNCHANGED: scaling / classical helpers
def fit_scale(X_train, X_test, bandwidth=1.0):
    scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
    scaler.fit(X_train)
    return scaler.transform(X_train) * bandwidth, scaler.transform(X_test) * bandwidth

def _param_combinations(grid):
    keys = list(grid)
    return [dict(zip(keys, vals)) for vals in itertools.product(*(grid[k] for k in keys))]

def _resolve_gamma(gamma, Xs):
    if gamma == 'scale':
        return 1.0 / (Xs.shape[1] * Xs.var()) if Xs.var() > 0 else 1.0
    return gamma

def _fit_svc(kernel_name, Xtr, ytr, params):
    gamma = _resolve_gamma(params.get('gamma', 'scale'), Xtr)
    if kernel_name == 'linear':
        clf = SVC(kernel='linear', C=params['C'])
    elif kernel_name == 'rbf':
        clf = SVC(kernel='rbf', C=params['C'], gamma=gamma)
    else:
        clf = SVC(kernel='poly', C=params['C'], degree=params['degree'], gamma=gamma, coef0=params['coef0'])
    return clf.fit(Xtr, ytr)

## 5. Splits, and how the training set is shrunk

`stratified_prefix_order` draws **one** random, class-stratified ordering of each outer-training fold. The training set of size *n*
is its first *n* peptides, so the sets are nested. The audit below proves the properties claimed in the introduction.

In [6]:
# ---------------------------------------------------------------- UNCHANGED: outer splits (group-aware, 5 x 20)
def repeated_stratified_group_kfold(y, groups, n_splits, n_repeats, random_state=RANDOM_SEED):
    splits = []
    for r in range(n_repeats):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state + r)
        splits.extend(sgkf.split(np.zeros(len(y)), y, groups))
    return splits

def make_outer_splits(y, groups, n_splits=N_SPLITS_OUTER, n_repeats=N_REPEATS_OUTER, random_state=RANDOM_SEED):
    return repeated_stratified_group_kfold(y, groups, n_splits, n_repeats, random_state)


# ---------------------------------------------------------------- NEW: how the training set is shrunk
def stratified_prefix_order(tr_idx, y, fold_i, seed=RANDOM_SEED):
    """One random ordering of the OUTER-TRAINING peptides per outer fold, stratified by class so that
    every prefix has (almost) the class proportions of the full fold. The training set of size n is the
    first n peptides of this ordering, so the sets are NESTED: 16 < 24 < 32 < ... < full."""
    rng = np.random.default_rng(seed + 7919 * fold_i)
    key = np.empty(len(tr_idx))
    for c in (0, 1):
        idx = np.where(y[tr_idx] == c)[0]
        key[idx] = (rng.permutation(len(idx)) + rng.random(len(idx))) / len(idx)
    return tr_idx[np.argsort(key)]

def _inner_splits(y_tr, groups_tr, fold_i, n_splits=N_SPLITS_INNER):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            return list(StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED + fold_i)
                        .split(np.zeros(len(y_tr)), y_tr, groups_tr))
        except ValueError:
            return []

def _usable(yi_tr, yi_val):
    """An inner fold is scored only if both its training and validation parts contain both classes."""
    return len(np.unique(yi_tr)) == 2 and len(np.unique(yi_val)) == 2

In [7]:
outer = make_outer_splits(y, groups)

audit = []
for size in TRAIN_SIZES:
    n_tr, pos_share, leaks, nested_ok = [], [], 0, True
    for fold_i, (tr_full, te_idx) in enumerate(outer):
        order = stratified_prefix_order(tr_full, y, fold_i)
        tr = tr_full if size == 'full' else np.sort(order[:size])
        n_tr.append(len(tr)); pos_share.append(y[tr].mean())
        leaks += bool(set(groups[tr]) & set(groups[te_idx])) or bool(set(tr) & set(te_idx))
        if size != 'full':
            nested_ok &= set(order[:size]) <= set(tr_full)
    audit.append({'size': str(size), 'mean_n_train': np.mean(n_tr), 'min_n_train': min(n_tr), 'max_n_train': max(n_tr),
                  'mean_share_strong': np.mean(pos_share), 'folds_with_leakage': leaks})
audit = pd.DataFrame(audit).set_index('size')
audit.to_csv(f'{OUTDIR}/training_set_audit.csv')
display(audit.round(3))
assert (audit['folds_with_leakage'] == 0).all(), 'a training subset shares a peptide or a group with its test fold'
print('Training sets have the intended size, keep the class balance, and never share a peptide or a group with the test fold.')

,mean_n_train,min_n_train,max_n_train,mean_share_strong,folds_with_leakage
size,,,,,
16,16.00,16,16,0.474,0
24,24.00,24,24,0.475,0
32,32.00,32,32,0.477,0
40,40.00,40,40,0.474,0
48,48.00,48,48,0.476,0
56,55.88,52,56,0.475,0
full,64.00,52,74,0.474,0


Training sets have the intended size, keep the class balance, and never share a peptide or a group with the test fold.


## 6. Nested CV at every training size

For each outer fold and each size: the inner loop (4 group-aware folds on the **subset only**) picks the hyperparameters, the model
is refitted on the whole subset, and it is evaluated **once** on that outer fold's untouched test peptides. An inner fold is
scored only if its training and validation parts both contain both classes; if no inner fold qualifies, a fixed default is used and
the row is flagged (`used_fallback`). The outer fold is never dropped.

In [8]:
# ---------------------------------------------------------------- NEW: nested CV at every training size
def learning_curve_nested(reps_data, y, groups, model, sizes=TRAIN_SIZES, outer=None,
                          n_splits_inner=N_SPLITS_INNER, fold_range=None, verbose=True):
    """model = ('quantum','zz'|'z') or ('classical','linear'|'rbf'|'poly').
    reps_data: dict name -> (N,d) array. One entry = fixed representation; several entries = the
    representation is chosen in the inner loop together with the hyperparameters (as in the final nested run).
    Returns (DataFrame with one row per (outer fold, size), predictions[size][fold])."""
    family, name = model
    if outer is None:
        outer = make_outer_splits(y, groups)
    combos = _param_combinations(CLASSICAL_GRIDS[name]) if family == 'classical' else None
    rows, preds = [], {}
    t0 = time.time()
    for fold_i, (tr_full, te_idx) in enumerate(outer):
        if fold_range is not None and not (fold_range[0] <= fold_i < fold_range[1]):
            continue
        order = stratified_prefix_order(tr_full, y, fold_i)
        for size in sizes:
            tr_idx = tr_full if size == 'full' else np.sort(order[:size])
            # ---- invariants: the outer test fold is untouched and group-disjoint from every training subset
            assert not (set(tr_idx) & set(te_idx))
            assert not (set(groups[tr_idx]) & set(groups[te_idx]))
            y_tr, y_te = y[tr_idx], y[te_idx]
            inner = _inner_splits(y_tr, groups[tr_idx], fold_i, n_splits_inner)

            inner_scores, n_valid = {}, 0
            for rep_name, X_all in reps_data.items():
                X_tr_outer = X_all[tr_idx]
                for in_tr, in_val in inner:
                    yi_tr, yi_val = y_tr[in_tr], y_tr[in_val]
                    if not _usable(yi_tr, yi_val):
                        continue
                    if rep_name == next(iter(reps_data)):
                        n_valid += 1
                    if family == 'quantum':
                        for bw in PARAM_GRID_QUANTUM['bandwidth']:
                            Xi_tr, Xi_val = fit_scale(X_tr_outer[in_tr], X_tr_outer[in_val], bandwidth=bw)
                            for reps in PARAM_GRID_QUANTUM['reps']:
                                K_tr, K_val = kernel_train_test(Xi_tr, Xi_val, name, reps)
                                for C in PARAM_GRID_QUANTUM['C']:
                                    clf = SVC(kernel='precomputed', C=C).fit(K_tr, yi_tr)
                                    s = balanced_accuracy_score(yi_val, clf.predict(K_val))
                                    inner_scores.setdefault((rep_name, reps, bw, C), []).append(s)
                    else:
                        Xi_tr, Xi_val = fit_scale(X_tr_outer[in_tr], X_tr_outer[in_val])
                        for ci, params in enumerate(combos):
                            clf = _fit_svc(name, Xi_tr, yi_tr, params)
                            s = balanced_accuracy_score(yi_val, clf.predict(Xi_val))
                            inner_scores.setdefault((rep_name, ci), []).append(s)

            used_fallback = not inner_scores
            first_rep = next(iter(reps_data))
            if used_fallback:                        # keep the outer fold; flag it
                if family == 'quantum':
                    best = (first_rep, FALLBACK_QUANTUM['reps'], FALLBACK_QUANTUM['bandwidth'], FALLBACK_QUANTUM['C'])
                else:
                    best_params = FALLBACK_CLASSICAL[name]; best = (first_rep, None)
            else:
                best = max(inner_scores, key=lambda k: np.mean(inner_scores[k]))

            row = {'fold': fold_i, 'size': str(size), 'n_train': len(tr_idx), 'n_test': len(te_idx),
                   'n_valid_inner': n_valid, 'used_fallback': used_fallback, 'best_representation': best[0]}
            X_best = reps_data[best[0]]
            if family == 'quantum':
                _, best_reps, best_bw, best_C = best
                Xtr, Xte = fit_scale(X_best[tr_idx], X_best[te_idx], bandwidth=best_bw)
                K_tr, K_te = kernel_train_test(Xtr, Xte, name, best_reps)
                y_pred = SVC(kernel='precomputed', C=best_C).fit(K_tr, y_tr).predict(K_te)
                row.update({'best_reps': best_reps, 'best_bandwidth': best_bw, 'best_C': best_C})
            else:
                if not used_fallback:
                    best_params = combos[best[1]]
                Xtr, Xte = fit_scale(X_best[tr_idx], X_best[te_idx])
                y_pred = _fit_svc(name, Xtr, y_tr, best_params).predict(Xte)
                row.update({f'best_{k}': v for k, v in best_params.items()})
            row.update({'accuracy': accuracy_score(y_te, y_pred),
                        'balanced_accuracy': balanced_accuracy_score(y_te, y_pred),
                        'mcc': matthews_corrcoef(y_te, y_pred) if len(set(y_pred)) > 1 else 0.0})
            rows.append(row)
            preds.setdefault(str(size), {})[fold_i] = {'test_idx': te_idx.tolist(), 'y_true': y_te.tolist(),
                                                       'y_pred': np.asarray(y_pred).tolist()}
        if verbose and fold_i % 10 == 0:
            print(f'   [{family}-{name}] fold {fold_i}/{len(outer)}  {time.time()-t0:5.0f}s')
    return pd.DataFrame(rows), preds

## 7. Uncertainty machinery (Nadeau-Bengio and group bootstrap, unchanged) and curve statistics

In [9]:
# ---------------------------------------------------------------- UNCHANGED: uncertainty machinery
def nadeau_bengio_ci(diff, n_train, n_test, alpha=0.05):
    diff = np.asarray(diff, dtype=float)
    J = len(diff)
    var = diff.var(ddof=1)
    se = np.sqrt(var * (1.0 / J + n_test / n_train))
    t = stats.t.ppf(1 - alpha / 2, df=J - 1)
    mean = diff.mean()
    t_stat = mean / se if se > 0 else np.nan
    p = 2 * (1 - stats.t.cdf(abs(t_stat), df=J - 1)) if np.isfinite(t_stat) else np.nan
    return float(mean), float(mean - t * se), float(mean + t * se), float(p)

def _peptide_tallies(predictions, n_samples):
    n_pred = np.zeros(n_samples); n_correct = np.zeros(n_samples)
    for fold_i in sorted(predictions, key=int):
        idx = np.asarray(predictions[fold_i]['test_idx'])
        yt = np.asarray(predictions[fold_i]['y_true']); yp = np.asarray(predictions[fold_i]['y_pred'])
        n_pred[idx] += 1
        n_correct[idx] += (yt == yp)
    return n_pred, n_correct

def _bacc_from_tallies(n_pred, n_corr, idx, pos):
    p, q = idx[pos[idx]], idx[~pos[idx]]
    tpr = n_corr[p].sum() / n_pred[p].sum() if n_pred[p].sum() > 0 else np.nan
    tnr = n_corr[q].sum() / n_pred[q].sum() if n_pred[q].sum() > 0 else np.nan
    return 0.5 * (tpr + tnr)

def bootstrap_group_ci(pred_a, pred_b, groups, y, n_boot=10000, seed=RANDOM_SEED):
    """UNCHANGED logic: paired bootstrap, resampling unit = sequence group."""
    n = len(y)
    na, ca = _peptide_tallies(pred_a, n); nb, cb = _peptide_tallies(pred_b, n)
    uniq = np.unique(groups); members = [np.where(groups == g)[0] for g in uniq]
    pos = (y == 1)
    rng = np.random.default_rng(seed); all_idx = np.arange(n)
    observed = _bacc_from_tallies(na, ca, all_idx, pos) - _bacc_from_tallies(nb, cb, all_idx, pos)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), size=len(uniq))])
        boots[b] = _bacc_from_tallies(na, ca, idx, pos) - _bacc_from_tallies(nb, cb, idx, pos)
    boots = boots[np.isfinite(boots)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(observed), float(lo), float(hi)


# ---------------------------------------------------------------- NEW: learning-curve statistics
def group_bootstrap_curve(pred_by_model, models, sizes, groups, y, n_boot=4000, seed=RANDOM_SEED,
                          contrasts=()):
    """ONE set of resampled groups is used for every model and every size, so every quantity below is
    paired. Returns the marginal balanced-accuracy CI per (model,size), the CI of every pairwise gap per size,
    and the CI of requested 'low-n minus high-n' contrasts of a gap.
    contrasts = [(model_a, model_b, low_sizes, high_sizes), ...]"""
    n = len(y); pos = (y == 1)
    tallies = {(m, str(s)): _peptide_tallies(pred_by_model[m][str(s)], n) for m in models for s in sizes}
    uniq = np.unique(groups); members = [np.where(groups == g)[0] for g in uniq]
    rng = np.random.default_rng(seed); all_idx = np.arange(n)

    def ba(m, s, idx):
        npred, ncorr = tallies[(m, str(s))]
        return _bacc_from_tallies(npred, ncorr, idx, pos)

    def stat(idx):
        b = {(m, str(s)): ba(m, s, idx) for m in models for s in sizes}
        c = {}
        for (a, bb, low, high) in contrasts:
            gap = lambda s: b[(a, str(s))] - b[(bb, str(s))]
            c[(a, bb)] = np.mean([gap(s) for s in low]) - np.mean([gap(s) for s in high])
        return b, c

    obs_b, obs_c = stat(all_idx)
    boot_b = {k: np.empty(n_boot) for k in obs_b}; boot_c = {k: np.empty(n_boot) for k in obs_c}
    for i in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), size=len(uniq))])
        bb_, cc_ = stat(idx)
        for k, v in bb_.items(): boot_b[k][i] = v
        for k, v in cc_.items(): boot_c[k][i] = v
    ci = lambda arr: tuple(np.nanpercentile(arr, [2.5, 97.5]))
    marg = {k: (obs_b[k],) + ci(boot_b[k]) for k in obs_b}
    gaps = {}
    for a in models:
        for bb in models:
            if a < bb:
                for s in sizes:
                    d = boot_b[(a, str(s))] - boot_b[(bb, str(s))]
                    gaps[(a, bb, str(s))] = (obs_b[(a, str(s))] - obs_b[(bb, str(s))],) + ci(d)
    cons = {k: (obs_c[k],) + ci(boot_c[k]) for k in obs_c}
    return marg, gaps, cons

## 8. Run B — the final nested procedure at every size (representation chosen in the inner loop)

Same procedure as `QSVM_Steve_Notes_nested.ipynb` (the representation is one more hyperparameter chosen **inside** the inner loop,
independently for each model), but applied to each training size, with the 45-value table excluded and the quantum kernels on.
It answers Dr. Rayan's point 9 without leakage: if the bias-variance explanation were right, the share of folds choosing a
**positional** representation should rise as the training set grows.

⚠️ This is the heavy run: the main nested run took about 45 minutes, and this repeats it at seven sizes (smaller sizes are cheaper), so budget
roughly 1.5-2.5 hours. Set `RUN_MODE_B = False` in section 1 to skip it.

In [14]:
MODELS = {'ZZFeatureMap': ('quantum', 'zz'), 'ZFeatureMap': ('quantum', 'z'),
          'Classical linear': ('classical', 'linear'), 'Classical RBF': ('classical', 'rbf'),
          'Classical poly': ('classical', 'poly')}
models = list(MODELS)
sizes = [str(s) for s in TRAIN_SIZES]
xs = [audit.loc[s, 'mean_n_train'] for s in sizes]

res_B, preds_B = {}, {}
if RUN_MODE_B:
    print('candidate representations:', list(representations))
    for label, spec in MODELS.items():
        t0 = time.time()
        res_B[label], preds_B[label] = learning_curve_nested(representations, y, groups, spec, outer=outer, verbose=False)
        res_B[label].to_csv(f'{OUTDIR}/runB_folds_{label.replace(" ", "_")}.csv', index=False)
        print(f'{label:18s} {time.time()-t0:6.0f}s   fallbacks: {int(res_B[label].used_fallback.sum())} / {len(res_B[label])}')
    json.dump(preds_B, open(f'{OUTDIR}/runB_predictions.json', 'w'))
else:
    print('Run B skipped (RUN_MODE_B = False)')

candidate representations: ['mean_z', 'z1_positional', 'pc1_positional_87', 'hopp_woods', 'kyte_doolittle', 'fauchere_pliska', 'euclid_centroid', 'mahalanobis_centroid']
ZZFeatureMap          854s   fallbacks: 0 / 700
ZFeatureMap         10813s   fallbacks: 0 / 700
Classical linear      123s   fallbacks: 0 / 700
Classical RBF         227s   fallbacks: 0 / 700
Classical poly        762s   fallbacks: 0 / 700


In [ ]:
import matplotlib.pyplot as plt

if RUN_MODE_B:
    share_rows = []
    for label, res in res_B.items():
        tab = res.groupby('size', sort=False)['best_representation'].value_counts(normalize=True).unstack(fill_value=0).loc[sizes]
        tab['model'] = label
        share_rows.append(tab.reset_index())
    share = pd.concat(share_rows, ignore_index=True)
    share.to_csv(f'{OUTDIR}/runB_selection_share_by_size.csv', index=False)
    print('Share of outer folds in which each representation was chosen (per model and training size):')
    display(share.round(2))

    mean_ba_B = pd.DataFrame({m: res_B[m].groupby('size', sort=False)['balanced_accuracy'].mean() for m in res_B}).loc[sizes]
    mean_ba_B.to_csv(f'{OUTDIR}/runB_mean_balanced_accuracy.csv')
    print('\nMean balanced accuracy by size, representation chosen in the inner loop:')
    display(mean_ba_B.round(3))

    fig, axes = plt.subplots(1, len(res_B), figsize=(3.2 * len(res_B), 3.2), sharey=True)
    for ax_, (label, res) in zip(np.atleast_1d(axes), res_B.items()):
        sh = res.assign(is_z=(res['best_representation'] == 'mean_z')).groupby('size', sort=False)['is_z'].mean().loc[sizes]
        ax_.plot(xs, sh.values, '-o'); ax_.set_title(label, fontsize=9); ax_.set_ylim(0, 1.02); ax_.grid(alpha=.25)
        ax_.set_xlabel('n_train')
    np.atleast_1d(axes)[0].set_ylabel('share of folds choosing mean z')
    plt.tight_layout(); plt.savefig(f'{OUTDIR}/exp9_final_runB_selection.png', dpi=130); plt.show()

## 9. Results of Run B: paired gaps and learning curves

Group bootstrap and Nadeau-Bengio intervals computed on the Run B predictions (representation chosen in the inner loop).


In [ ]:
if RUN_MODE_B:
    chk = res_B[models[0]].groupby('size', sort=False).agg(n_train=('n_train', 'mean'), usable_inner_folds=('n_valid_inner', 'mean'),
                                                           outer_folds=('fold', 'count'))
    print('Design check (every size keeps all 100 outer folds):'); display(chk.loc[sizes].round(2))

    # group bootstrap: ONE set of resampled groups for every model and size -> all differences are paired
    contrasts = [(a, 'Classical linear', ['16', '24'], ['48', '56', 'full'])
                 for a in ['Classical RBF', 'Classical poly', 'ZFeatureMap', 'ZZFeatureMap']] + \
                [(a, 'Classical RBF', ['16', '24'], ['48', '56', 'full']) for a in ['ZFeatureMap', 'ZZFeatureMap']]
    marg, gaps, cons = group_bootstrap_curve(preds_B, models, sizes, groups, y, n_boot=4000, contrasts=contrasts)

    def gap(a, b, s):
        if (a, b, s) in gaps: return gaps[(a, b, s)]
        o, lo, hi = gaps[(b, a, s)]
        return (-o, -hi, -lo)

    pairs = [('Classical RBF', 'Classical linear'), ('Classical poly', 'Classical linear'),
             ('ZFeatureMap', 'Classical linear'), ('ZZFeatureMap', 'Classical linear'),
             ('ZFeatureMap', 'Classical RBF'), ('ZZFeatureMap', 'Classical RBF'), ('ZZFeatureMap', 'ZFeatureMap')]
    rows = []
    for a, b in pairs:
        for s in sizes:
            da = res_B[a][res_B[a]['size'] == s].sort_values('fold')
            db = res_B[b][res_B[b]['size'] == s].sort_values('fold')
            d = da['balanced_accuracy'].values - db['balanced_accuracy'].values
            m, lo, hi, p = nadeau_bengio_ci(d, da['n_train'].mean(), da['n_test'].mean())
            go, glo, ghi = gap(a, b, s)
            rows.append({'comparison': f'{a} - {b}', 'size': s, 'mean_diff': m, 'NB_lo': lo, 'NB_hi': hi, 'NB_excl0': lo > 0 or hi < 0,
                         'group_lo': glo, 'group_hi': ghi, 'group_excl0': glo > 0 or ghi < 0})
    gap_df = pd.DataFrame(rows)
    gap_df['jointly_significant'] = gap_df['NB_excl0'] & gap_df['group_excl0']
    gap_df.to_csv(f'{OUTDIR}/runB_paired_gaps.csv', index=False)

    print('Paired gaps in balanced accuracy, per training size.  A gap is claimable only if it is JOINTLY significant (both intervals exclude 0).')
    display(gap_df.round(3))
    print(f"\nJointly significant gaps: {int(gap_df['jointly_significant'].sum())} of {len(gap_df)}")

    print('\nContrast of each gap, small n (16, 24) minus large n (48, 56, full). Positive = the model gains MORE over its reference at small n.')
    con_df = pd.DataFrame([{'gap': f'{a} - {b}', 'contrast': o, 'group_lo': lo, 'group_hi': hi, 'excludes_0': lo > 0 or hi < 0}
                           for (a, b), (o, lo, hi) in cons.items()])
    con_df.to_csv(f'{OUTDIR}/runB_low_vs_high_n_contrasts.csv', index=False)
    display(con_df.round(3))
    print('These contrasts use the group bootstrap only (no Nadeau-Bengio analogue exists for a difference of gaps): treat them as exploratory.')

In [ ]:
if RUN_MODE_B:
    import matplotlib.pyplot as plt
    mk = {k: v for k, v in marg.items()}
    xs = [audit.loc[s, 'mean_n_train'] for s in sizes]
    col = {'Classical linear': '#1f77b4', 'Classical RBF': '#2ca02c', 'Classical poly': '#8c564b', 'ZFeatureMap': '#ff7f0e', 'ZZFeatureMap': '#d62728'}
    order_m = ['Classical linear', 'Classical RBF', 'Classical poly', 'ZFeatureMap', 'ZZFeatureMap']

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
    for i, m in enumerate(order_m):
        v = np.array([mk[(m, s)] for s in sizes])
        ax[0].errorbar(np.array(xs) + (i - 2) * 0.35, v[:, 0], yerr=[v[:, 0] - v[:, 1], v[:, 2] - v[:, 0]],
                       fmt='o', ms=4, capsize=2, color=col[m], label=m)
    ax[0].axhline(0.5, color='gray', ls=':', lw=1); ax[0].text(16, 0.51, 'majority baseline', color='gray', fontsize=8)
    ax[0].set_xlabel('training-set size (n_train)'); ax[0].set_ylabel('balanced accuracy (outer test)')
    ax[0].set_title('Learning curves, representation chosen in the inner loop\n(points: mean over 100 outer folds; bars: group-bootstrap 95% CI)', fontsize=9)
    ax[0].legend(fontsize=8, loc='lower right'); ax[0].grid(alpha=.25)

    for i, a in enumerate(['Classical RBF', 'Classical poly', 'ZFeatureMap', 'ZZFeatureMap']):
        v = np.array([gap(a, 'Classical linear', s) for s in sizes])
        ax[1].errorbar(np.array(xs) + (i - 1.5) * 0.4, v[:, 0], yerr=[v[:, 0] - v[:, 1], v[:, 2] - v[:, 0]],
                       fmt='s', ms=4, capsize=2, color=col[a], label=f'{a} - linear')
    ax[1].axhline(0, color='k', lw=1); ax[1].set_ylim(-0.135, 0.06)
    ax[1].set_xlabel('training-set size (n_train)'); ax[1].set_ylabel('paired gap in balanced accuracy vs. linear')
    ax[1].set_title('Is non-linearity an advantage at small n?\n(group-bootstrap 95% CI; above 0 = better than linear)', fontsize=9)
    ax[1].legend(fontsize=7.5, loc='upper left', ncol=2); ax[1].grid(alpha=.25)
    plt.tight_layout(); plt.savefig(f'{OUTDIR}/exp9_final_runB_learning_curves.png', dpi=130); plt.show()

## 10. How to read this, and what was verified

* **Claim only what is jointly significant.** The group bootstrap resamples peptides' groups but not the training-set draw or the tuning, so it is the narrower
  interval; Nadeau-Bengio is the conservative one. The report's rule (a difference counts only if both exclude 0) applies to every row above.
* The contrasts in section 9 test the *trend* in a gap (the actual hypothesis) but only with the group bootstrap, so they are exploratory.
* The learning curves have not flattened at the largest size: more peptides would probably still help, which supports obtaining new peptides rather than more seeds.

**Verification performed when this notebook was written** (reference environment: scikit-learn 1.8, qiskit 2.x):
1. `learning_curve_nested(..., sizes=['full'])` was compared with the **unmodified** `nested_cv_classical` / `nested_cv_quantum` of the nested notebook,
   restricted to the mean z-scale: identical predictions in all 100 outer folds for ZZFeatureMap, ZFeatureMap, linear, RBF and polynomial.
2. Every (size, fold) pair was kept: 3500 rows (5 models x 7 sizes x 100 folds), 0 fallbacks, and training sizes are exact.
3. `mean_z` was rebuilt from `zscaled_5_45.csv` in that environment; the notebook re-checks it against `zscale_dataset.csv` in section 2.

In [16]:
import sklearn, scipy
manifest = {'qiskit_version': None, 'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
            'sklearn': sklearn.__version__, 'random_seed': RANDOM_SEED,
            'cv': {'outer': [N_SPLITS_OUTER, N_REPEATS_OUTER], 'inner': N_SPLITS_INNER, 'group_aware': True,
                   'training_sizes': [str(s) for s in TRAIN_SIZES],
                   'subsampling': 'class-stratified nested prefixes of the outer-training fold (individual peptides)'},
            'quantum_grid': PARAM_GRID_QUANTUM,
            'classical_grids': {k: {kk: [str(x) for x in vv] for kk, vv in v.items()} for k, v in CLASSICAL_GRIDS.items()},
            'fallbacks': {'classical': FALLBACK_CLASSICAL, 'quantum': FALLBACK_QUANTUM},
            'n_groups': int(len(np.unique(groups))), 'representations_available': {k: int(v.shape[1]) for k, v in representations.items()},
            'run_B_executed': bool(RUN_MODE_B)}
try:
    import qiskit; manifest['qiskit_version'] = qiskit.__version__
except ImportError:
    pass
json.dump(manifest, open(f'{OUTDIR}/run_manifest.json', 'w'), indent=2)
print(json.dumps(manifest, indent=2)[:1500])

{
  "qiskit_version": "1.4.5",
  "numpy": "1.26.4",
  "pandas": "2.3.2",
  "scipy": "1.15.3",
  "sklearn": "1.7.2",
  "random_seed": 42,
  "cv": {
    "outer": [
      5,
      20
    ],
    "inner": 4,
    "group_aware": true,
    "training_sizes": [
      "16",
      "24",
      "32",
      "40",
      "48",
      "56",
      "full"
    ],
    "subsampling": "class-stratified nested prefixes of the outer-training fold (individual peptides)"
  },
  "quantum_grid": {
    "reps": [
      1,
      2
    ],
    "bandwidth": [
      0.05,
      0.1,
      0.2,
      0.5
    ],
    "C": [
      0.1,
      1,
      10
    ]
  },
  "classical_grids": {
    "linear": {
      "C": [
        "0.1",
        "1",
        "10"
      ]
    },
    "rbf": {
      "C": [
        "0.1",
        "1",
        "10"
      ],
      "gamma": [
        "scale",
        "0.01",
        "0.1",
        "1"
      ]
    },
    "poly": {
      "C": [
        "0.1",
        "1",
        "10"
      ],
      "degree": 